# CNN com Optuna — Arquitetura FLEXIVEL + Splits ESTRATIFICADOS (LRG)

Notebook de spec-z pra **LRG** (Luminous Red Galaxies) do eBOSS, usando CNN 1D com Optuna pra busca de hiperparametros.

## O que este notebook faz

1. **Carrega espectros** padded do HDF5 (~166k LRGs, 4635 pontos cada apos o padding feito no notebook 02)
2. **Splita estratificado** em train / val / test (preserva distribuicao de z em cada split)
3. **Otimiza arquitetura + HPs com Optuna**: numero de blocos conv, dense, kernel, dropout, lr, batch, etc.
4. **Retreina o melhor modelo** por mais epocas e avalia no test (intocado)
5. **Salva** modelo, predicoes, metadados, plots

## Decisao cientifica: spec-z so pela FORMA do espectro

A CNN recebe **apenas o espectro normalizado** (max-abs por linha → fica em [-1, 1]). Isso e proposital:

- Spec-z deve depender da **forma** do espectro (linhas, breaks, continuum shape), nao do brilho absoluto.
- Brilho absoluto depende de calibracao instrumental e selection effects do eBOSS — viesa o modelo a algo nao-fisico.
- Modelo so com forma generaliza melhor pra outras surveys (DESI, SDSS).

### E os scalars (log_max, log_median, log_sum, log_p95)?

Calculo e **salvo** essas features por espectro (em log10 do fluxo bruto), mas elas **NAO entram no modelo**. Servem so pra:
- Sanity check: ver se brilho correlaciona com z (deve dar `corr < 0`, galaxias mais fracas em z maior)
- Identificacao de outliers (galaxias atipicas tem scalars atipicos)
- Comparacao entre objetos depois do treino

## Splits ESTRATIFICADOS (anti-leak)

Em vez de split aleatorio (que pode dar test com z muito diferente de train), discretizo z em **20 bins de quantis** e splito **preservando a proporcao de cada bin** em cada fatia:

- **test (~15%)**: TRANCADO ate o final. Numeros honestos so saem dele.
- **val (~13%)**: usado pelo Optuna (cada trial avalia aqui) e pelo EarlyStopping no retreino.
- **train (~72%)**: onde o modelo realmente aprende.

`normalize_spectra` (max-abs por linha) e **stateless** — sem leak entre amostras. Scalars idem (per-linha).

## HPs que o Optuna escolhe

| Tipo | HPs |
|---|---|
| **Topologia** | `n_conv_blocks` (1-4), `n_dense_layers` (1-4), `n_units` (128-1024) |
| **Conv** | `kernel_size` ([11, 15, 21, 25]) |
| **Regularizacao** | `dropout_rate` (0.05-0.35), `weight_decay` (2e-6 a 5e-3) |
| **Otimizacao** | `learning_rate` (1e-4 a 1e-2), `batch_size` ([32, 64, 128, 256]) |
| **Ativacao** | `activation` (relu/elu/swish) |

Filtros das convs, decay do kernel e schedule de dropout sao **fixos** (convencoes solidas de CNN — nao vale gastar trials explorando).

## Score do Optuna: MAE simples

Cada trial e avaliado pelo `MAE(z_pred, z_true)` no val. Pra spec-z (precisao ~10x melhor que photo-z), MAE simples discrimina bem entre trials. A funcao `composite_objective` (sigma_NMAD + bias + outliers) tambem esta disponivel via `OBJECTIVE_MODE="composite"` .

## Robustez

- **SQLite + `load_if_exists=True`**: se o job cair (time limit, OOM), basta resubmeter — continua de onde parou.
- **PercentilePruner**: mata trials no pior 30% cedo, economizando tempo.
- **`catch=(ResourceExhaustedError,)`**: OOMs viram FAIL no Optuna, nao matam o job.
- **`RUN_TAG` com hash MD5 da config**: mudou qualquer coisa, vira run novo (nao sobrescreve).



In [ ]:
# ============================================================
# PAPERMILL PARAMETERS
# ============================================================
# Tag `parameters` permite executar via:
#   papermill cnn_optuna_qso_flex.ipynb out.ipynb -p N_TRIALS 100 -p SEED 7

OBJECT_TYPE      = "QSO"      # "ELG" | "LRG" | "QSO"
SEED             = 42         # reprodutibilidade
N_TRIALS         = 50         # trials do Optuna
EPOCHS_PER_TRIAL = 40         # max epocas por trial (pruner corta os ruins cedo)
FINAL_EPOCHS     = 120        # epocas no retreino final
OBJECTIVE_MODE   = "mae"          # "mae" (default, simples) | "composite" (sigma + bias + outliers, photo-z legacy)
USE_Z_WEIGHTS    = False      # se True, pesa amostras por faixa de z
TARGET_TRIALS_OVERRIDE = 0    # 0 usa N_TRIALS. valor maior estende um study existente ate esse alvo, sem mudar o config_hash


## 1. Config em dicionarios + hash MD5

Toda a config (training, objective, HPO, HP_SPACE, scalars, weights) vira um dicionario serializavel.
O **hash MD5** desse dicionario vai no `RUN_TAG` → se voce mudar **qualquer coisa**, vira um run novo.



In [ ]:
# ============================================================
# TRAINING CONFIG
# ============================================================
TRAINING_CONFIG = {
    "epochs":                 EPOCHS_PER_TRIAL,
    "test_size":              0.15,    # fracao do total que vira test
    "val_size":               0.15,    # fracao do POOL que vira val
    "early_stop_patience":    20,
    "early_stop_min_delta":   0.0,      # 0 = igual baseline. O antigo 5e-4 era > loss convergida (~1.8e-4) e congelava o treino em mae ~0.02
    "reduce_lr_factor":       0.3,      # alinhado ao baseline (era 0.5)
    "reduce_lr_patience":     10,     # alinhado ao baseline (era 5) — nao derruba o lr cedo demais
    "reduce_lr_min":          1e-8,     # alinhado ao baseline (era 1e-7)
}


# ============================================================
# OBJECTIVE CONFIG
# ============================================================
OBJECTIVE_CONFIG = {
    "mode":     OBJECTIVE_MODE,
    "w_sigma":  0.5,
}


# ============================================================
# HPO CONFIG (Optuna)
# ============================================================
HPO_CONFIG = {
    "target_trials":     N_TRIALS,
    "n_startup_trials":  10,
    "n_warmup_steps":    3,
    "interval_steps":    1,
    "pruner_percentile": 75, #entre 70 e 80 costuma ser bom pra cortar os piores ~30% dos trials a cada etapa, mas nao muito agressivo a ponto de cortar bons candidatos cedo demais
}


# ============================================================
# HP_SPACE — espaco de busca
# ============================================================
HP_SPACE = {
    # ARQUITETURA
    "n_conv_blocks":  {"low": 1, "high": 4},
    "n_dense_layers": {"low": 1, "high": 4},
    "n_units":        {"low": 128, "high": 1024, "step": 64},
    "kernel_size":    [11, 15, 21, 25],
    "use_batchnorm":  [True, False],   # baseline (0.0038) NAO usa BN; deixa o Optuna decidir
    "dropout_schedule": ["ramp", "flat"],  # ramp = cresce por bloco (flex atual); flat = leve e constante (estilo baseline)
    "taper_last_block": [True, False],  # True = 4o bloco afunila (1 conv, filtros/2, sem pool/dropout) igual baseline

    # HPs CLASSICOS
    "dropout_rate":   {"low": 0.05, "high": 0.35},
    "learning_rate":  {"low": 1e-4, "high": 3e-3, "log": True},
    "weight_decay":   {"low": 2e-6, "high": 5e-3, "log": True},
    "activation":     ["relu", "elu", "swish"],
    "batch_size":     [32, 64, 128, 256],
    # LEVERS DE PIPELINE (nao so arquitetura) — deixam o Optuna recuperar a receita do baseline
    "optimizer":      ["adamw", "adam"],   # adamw = pipeline flex (weight decay); adam = pipeline do baseline (sem wd)
}


# ============================================================
# SCALARS CONFIG — calculados e salvos PRA ANALISE, NAO usados no modelo
# ============================================================
# Computadas no espectro BRUTO (antes da normalize_spectra), ficam em log10
# puro (fisicamente interpretavel). NAO alimentam o modelo — decisao
# cientifica (ver intro do notebook).
SCALARS_CONFIG = {
    "features":   ["log_max", "log_median", "log_sum", "log_p95"],  # ver extract_scalars
    "use_log10":  True,         # log10 pra colapsar ordens de magnitude
    "mask_nonpositive": True,   # ignora padding (zeros) e fluxos negativos (ruido) ao computar median/p95
    "used_in_model":   False,   # so calculados e salvos pra ANALISE — nao alimentam o modelo
}


# ============================================================
# Z-WEIGHTS — pesos opcionais por faixa de z
# ============================================================
WEIGHTS_CONFIG = {
    "use_z_weights": USE_Z_WEIGHTS,
    "z_focus_low":   0.85,
    "z_focus_high":  1.00,
    "w_low":          1.0,
    "w_focus":        2.0,
    "w_high":         0.7,
    "w_max":          3.0,
}


# ============================================================
# HASH MD5 DA CONFIG — id curto e estavel do experimento
# ============================================================
# Por que MD5?
#   - Identifica unicamente UMA combinacao especifica de TODOS os parametros.
#   - Mudou QUALQUER coisa (lr range, dropout, seed, scalars on/off) -> hash muda.
#   - 8 chars eh suficiente pra ~4 bilhoes de combinacoes (colisao improvavel).
#
# Por que JSON antes de hashear?
#   - hashlib precisa de bytes. dict nao eh hasheavel direto.
#   - sort_keys=True: ordem de chaves nao afeta o hash (resultado deterministico).
#   - separators=(",", ":"): formato compacto sem espacos (evita variacao de
#     hash so por whitespace).
#   - .encode("utf-8"): str -> bytes.
#
# import "as _json" (com underscore): convencao pra evitar shadow do nome
# "json" se outra cell importar como "json" tambem.
import hashlib, json as _json

# Coleta TUDO que define este experimento num unico dict
config_payload = {
    "training":   TRAINING_CONFIG,
    "objective":  OBJECTIVE_CONFIG,
    "hpo":        HPO_CONFIG,
    "hp_space":   HP_SPACE,
    "scalars":    SCALARS_CONFIG,
    "weights":    WEIGHTS_CONFIG,
    "seed":       SEED,
    "object":     OBJECT_TYPE,
}

# Pipeline: dict -> JSON string -> bytes UTF-8 -> MD5 hex -> primeiros 8 chars
config_hash = hashlib.md5(
    _json.dumps(config_payload, sort_keys=True, separators=(",", ":")).encode("utf-8")
).hexdigest()[:8]

print(f"config_hash = {config_hash}")
print(f"objective   = {OBJECTIVE_MODE}")
print(f"scalars     = {SCALARS_CONFIG['features']}")
print(f"z_weights   = {USE_Z_WEIGHTS}")


## 2. Imports + reprodutibilidade


In [ ]:
# ============================================================
# CONFIG ANTES DE IMPORTAR TF (logs mais legiveis)
# ============================================================
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import sys, time, random, platform
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
import keras
import sklearn
from sklearn.model_selection import StratifiedShuffleSplit
import optuna


# ============================================================
# WALK-UP DO PROJECT_ROOT
# ============================================================
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR
while not (PROJECT_ROOT / "config.py").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

# ============================================================
# IMPORTS DO PROJETO — codigo escrito no nosso repo
# ============================================================
# config.py define onde estao dados / resultados / modelos (paths centralizados).
#   ARQUIVO: /home/thalita/spec_z_ml/config.py
#   - paths_for(OBJECT_TYPE): retorna dict com paths do objeto (HDF5, splits, etc.)
#   - RESULTS_DIR, MODELS_DIR: Path objects pros diretorios raiz de saidas
from config import paths_for, RESULTS_DIR, MODELS_DIR

# src/data/loader.py contem leitura de HDF5 + normalizacao max-abs.
#   ARQUIVO: /home/thalita/spec_z_ml/src/data/loader.py
#   - load_spectral_dataset(hdf5_path): le /ml_dataset/X_spec e /ml_dataset/y
#   - normalize_spectra(X): divide cada espectro pelo proprio max(|x|) — stateless,
#                          per-linha, sem leak entre amostras.
from src.data import load_spectral_dataset, normalize_spectra

# src/models/cnn.py tem layer custom ScaledSoftplus (saida z >= 0 sem teto).
#   ARQUIVO: /home/thalita/spec_z_ml/src/models/cnn.py
#   - softplus(beta*x)/beta: smooth, monotonica, sempre positiva
#   - registrada via @keras.utils.register_keras_serializable -> sobrevive ao
#     model.save()/load_model() sem precisar de custom_objects
from src.models.cnn import ScaledSoftplus


# ============================================================
# REPRODUTIBILIDADE
# ============================================================
def set_reproducibility(seed: int = 42):
    """Liga seeds em python, numpy, tf/keras pra resultados reprodutiveis."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    keras.utils.set_random_seed(seed)
    tf.random.set_seed(seed)

set_reproducibility(SEED)

sns.set_theme(style="whitegrid", palette="deep")


In [ ]:
# ============================================================
# RUNTIME SANITY CHECK — versoes + benchmark rapido da GPU
# ============================================================
# Reporta versoes de tudo (importante pra reprodutibilidade — diferentes
# versoes podem dar resultados ligeiramente diferentes).
print("=== Runtime ===")
# .split()[0]: pega so o numero da versao, descarta o resto (ex: "3.13.0 (...)")
print(f"python    : {sys.version.split()[0]} | {platform.platform()}")
print(f"tf        : {tf.__version__} | keras: {keras.__version__}")
print(f"numpy     : {np.__version__} | sklearn: {sklearn.__version__} | optuna: {optuna.__version__}")

# Lista GPUs disponiveis. Se rodando em CPU (login node), retorna [].
gpus = tf.config.list_physical_devices("GPU")
print(f"\nGPUs visiveis: {len(gpus)}  {gpus}")

# Benchmark rapido: multiplicacao de matriz 1024x1024.
#   - Em GPU: ~0.2-0.5s
#   - Em CPU: ~5-20s
# Se sua GPU "deveria estar la" mas o matmul demora segundos, algo errado
# (driver, CUDA, etc). Outra utilidade: aquece a GPU antes do treino real.
#
# .numpy() forca execucao eager imediata (sem .numpy(), TF poderia adiar).
# tf.random.normal: amostra normal padrao N(0, 1).
t0 = time.time()
_ = tf.matmul(tf.random.normal((1024, 1024)), tf.random.normal((1024, 1024))).numpy()
# ternario inline: 'GPU' se gpus eh truthy (lista nao-vazia), senao 'CPU'
print(f"matmul 1024x1024: {time.time()-t0:.2f}s ({'GPU' if gpus else 'CPU'})")


## 3. Carregar dados + extrair scalars + normalizar

### Ordem importa
1. **Carrega** o espectro bruto do HDF5 (sem normalizar).
2. **Extrai scalars** do bruto (`log_max`, `log_median`, `log_sum`, `log_p95`) — essa info se perderia se normalizassemos antes.
3. **Normaliza** o espectro com `normalize_spectra` (max-abs por linha → fica em [-1, 1]).

### `extract_scalars` — features de escala global

Para cada espectro:
- `log_max`     = log10(max(|x|))          — pico de fluxo (luminosidade)
- `log_median`  = log10(median(|x_pos|))   — nivel do continuum (so valores positivos)
- `log_sum`     = log10(sum(|x_pos|))      — fluxo total integrado (brilho)
- `log_p95`     = log10(percentil 95 de |x_pos|) — "pico robusto" (menos ruidoso que max)

Mascaramos valores nao-positivos (padding zeros + ruido negativo) antes de median/p95 pra nao distorcer.



In [ ]:
def extract_scalars(X_orig):
    """Features de escala por espectro, computadas no fluxo BRUTO (pre-normalizacao).

    Retorna (N, 4) com colunas: log_max, log_median, log_sum, log_p95.
    Usa log10 pra colapsar ordens de magnitude. Mascara valores nao-positivos
    (padding + ruido negativo) ao computar median e p95.

    Sem eps clamp: fluxos astronomicos podem ser ~1e-17 (erg/s/cm^2/A), bem
    abaixo de qualquer eps razoavel. Em vez disso, deixa log10 ir pra -inf
    no edge case (espectro all-zero) e substitui por -40 (sentinela).
    """
    X_abs = np.abs(X_orig)

    # Pra median e p95: ignora padding/zeros (NaN-aware)
    X_pos = np.where(X_orig > 0, X_orig, np.nan)

    with np.errstate(divide="ignore", invalid="ignore"):
        # log_max — pico de fluxo (linha mais forte em ELG/QSO; pico do continuum em LRG).
        # proxy do "brilho instantaneo" maximo do espectro.
        log_max    = np.log10(X_abs.max(axis=1))

        # log_median — nivel tipico do continuum (mediana ignora picos de linha e outliers).
        # galaxia brilhante tem continuum alto; galaxia fraca, continuum baixo.
        log_median = np.log10(np.nanmedian(X_pos, axis=1))

        # log_sum — fluxo integrado em todos os pontos. proxy de luminosidade
        # observada TOTAL. galaxia mais distante (z maior) tipicamente tem sum menor
        # pela lei do inverso do quadrado da distancia.
        log_sum    = np.log10(X_abs.sum(axis=1))

        # log_p95 — "pico robusto" (top 5% do fluxo). igual ao max em essencia, mas
        # ignora 1-2 spikes — mais estavel quando ha cosmic rays nao removidos.
        log_p95    = np.log10(np.nanpercentile(X_pos, 95, axis=1))

    scalars = np.stack([log_max, log_median, log_sum, log_p95], axis=1)
    # -inf (log de zero) e NaN (espectro all-zero) viram -40 (sentinela)
    scalars = np.where(np.isfinite(scalars), scalars, -40.0)
    return scalars.astype(np.float32)


# ============================================================
# CARREGA, EXTRAI, NORMALIZA — nessa ordem
# ============================================================
paths = paths_for(OBJECT_TYPE)
HDF5_PATH = paths["spectra_h5"].with_name(f"{OBJECT_TYPE}spectra_padded.h5")
print(f"HDF5: {HDF5_PATH}")

X_orig, y_all, n_wave = load_spectral_dataset(HDF5_PATH)        # bruto

scalars_all = extract_scalars(X_orig)                            # ANTES de normalizar
n_scalars   = scalars_all.shape[1]
SCALARS_CONFIG_FEATURES = SCALARS_CONFIG["features"]

X_spec = normalize_spectra(X_orig).astype(np.float32)            # agora normaliza
y_all  = y_all.astype(np.float32)

del X_orig   # libera memoria (ja extraimos o que precisavamos)


print(f"\nX_spec     : {X_spec.shape}  (espectros normalizados, max-abs por linha)")
print(f"scalars_all: {scalars_all.shape}  (n_scalars = {n_scalars})")
print(f"y_all      : {y_all.shape}  z range [{y_all.min():.3f}, {y_all.max():.3f}]")

# Sanity check: ver range dos scalars (em log10) — devem ser numeros tipo 1-5
print(f"\nRange dos scalars (em log10):")
for i, name in enumerate(SCALARS_CONFIG_FEATURES):
    s = scalars_all[:, i]
    print(f"  {name:>12} : min={s.min():.3f}  median={np.median(s):.3f}  max={s.max():.3f}")


## Sanity check: scalars correlacionam com z?

Antes de splitar, vale ver se os 4 scalars **carregam informacao sobre z**. Se a correlacao for forte (|corr| > 0.3), o modelo vai gostar dessas features. Se for zero, talvez ajudem mais identificando **tipo** de galaxia que distancia.

Galaxias mais distantes (z maior) tendem a parecer mais fracas (lei do inverso do quadrado da distancia) — espera-se que `log_max`, `log_sum`, `log_p95` correlacionem **negativamente** com z.


In [ ]:
# ============================================================
# SCALARS vs z — hexbin de cada feature + correlacao com redshift
# ============================================================
# Pra cada uma das 4 features (log_max, log_median, log_sum, log_p95) faz:
#   - um hexbin 2D mostrando a distribuicao conjunta (scalar, z)
#   - calcula a correlacao linear de Pearson com z
#
# Hexbin = histograma 2D em celulas hexagonais. Cor escura = muitas galaxias
# naquele "pixel". Melhor que scatter quando ha muitas amostras (>10k) — scatter
# vira uma mancha unica e nao da pra ver a densidade.

# Cria figura com 4 paineis lado a lado
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

# Itera sobre cada feature: i = indice da coluna, name = nome ('log_max', etc.)
for i, name in enumerate(SCALARS_CONFIG_FEATURES):
    # Pega a coluna i de scalars_all (valores dessa feature pra TODAS as galaxias)
    s = scalars_all[:, i]

    # np.corrcoef devolve uma matriz 2x2 de correlacoes:
    #   [[corr(y,y), corr(y,s)],
    #    [corr(s,y), corr(s,s)]]
    # A correlacao que queremos eh corr(y, s) — elemento [0, 1].
    # Valores entre -1 (anticorrelacao perfeita) e +1 (correlacao perfeita).
    corr = np.corrcoef(y_all, s)[0, 1]

    # Hexbin: x=z, y=valor da feature
    #   gridsize=40 → 40 hexagonos no eixo maior
    #   cmap="viridis" → escala perceptualmente uniforme
    #   mincnt=1 → so colore celulas com >= 1 galaxia (deixa fundo branco)
    hb = axes[i].hexbin(y_all, s, gridsize=40, cmap="viridis", mincnt=1)

    axes[i].set_xlabel("z")
    axes[i].set_ylabel(f"{name} (log10)")
    # +.3f imprime sempre com sinal (+/-) e 3 casas decimais
    axes[i].set_title(f"{name}  (corr = {corr:+.3f})")

    # Colorbar mostra a contagem de galaxias por celula hexagonal
    plt.colorbar(hb, ax=axes[i], shrink=0.8)

plt.tight_layout()
plt.show()


# ============================================================
# TABELA — correlacao + classificacao verbal
# ============================================================
# Convencao informal:
#   |corr| > 0.3   → "forte"  (feature claramente carrega info de z)
#   |corr| > 0.1   → "media"  (alguma info, mas nao dominante)
#   resto          → "fraca"  (feature pode ainda ajudar identificando TIPO de galaxia
#                              em vez de distancia direta)
print(f"\nCorrelacao linear (Pearson) com z:")
for i, name in enumerate(SCALARS_CONFIG_FEATURES):
    corr = np.corrcoef(y_all, scalars_all[:, i])[0, 1]
    interp = "forte" if abs(corr) > 0.3 else ("media" if abs(corr) > 0.1 else "fraca")
    # {:>12} alinha o nome a direita em 12 caracteres → fica em coluna
    print(f"  {name:>12} : {corr:+.4f}  ({interp})")


## 4. Splits 3-way ESTRATIFICADOS

### Estratificacao por z
- Discretiza z em bins por quantis (com fallback automatico se algum bin for muito pequeno).
- Splits 1 e 2 usam `StratifiedShuffleSplit` para preservar a proporcao de cada bin.

Os scalars sao splitados junto (mesmas amostras vao pra mesmo split), mas nao sao mais z-scoreados — ficam em log10 puro (fisicamente interpretavel). Eles servem so pra analise, nao alimentam o modelo.



In [ ]:
# ============================================================
# SPLIT — canonico ESTRATIFICADO por z (splits/<OBJ>_split.npz)
# ============================================================
# Fonte unica do projeto: src/data/splits.py (mesma politica estratificada
# que era feita aqui inline). TODOS os modelos compartilham o MESMO test set.
from config import SPLITS_DIR
from src.data import make_or_load_split, make_strat_bins

train_idx, val_idx, test_idx = make_or_load_split(OBJECT_TYPE, y_all, SPLITS_DIR)

# Reconstroi pool + indices relativos pra manter compatibilidade com split_idx.npz
# (consumido por notebooks que reproduzem o mesmo split, ex.: cnn_*_stratified).
pool_idx      = np.concatenate([train_idx, val_idx])
train_in_pool = np.arange(len(train_idx))
val_in_pool   = np.arange(len(train_idx), len(pool_idx))

y_pool = y_all[pool_idx]
# q efetivo dos bins de z (so' pra registro em run_info)
_, q_outer = make_strat_bins(y_all)
_, q_inner = make_strat_bins(y_pool)

X_train, X_val, X_test = X_spec[train_idx], X_spec[val_idx], X_spec[test_idx]
y_train, y_val, y_test = y_all[train_idx],  y_all[val_idx],  y_all[test_idx]
S_train, S_val, S_test = scalars_all[train_idx], scalars_all[val_idx], scalars_all[test_idx]


# ============================================================
# RESHAPE — Conv1D espera (N, n_wave, 1)
# ============================================================
X_train = X_train.reshape(-1, n_wave, 1)
X_val   = X_val.reshape(-1, n_wave, 1)
X_test  = X_test.reshape(-1, n_wave, 1)


# ============================================================
# CONFERIR — splits devem ter z parecido (sanity da estratificacao)
# ============================================================
print(f"Train : n = {len(y_train):>7,}  z_mean = {y_train.mean():.4f}  z_std = {y_train.std():.4f}")
print(f"Val   : n = {len(y_val):>7,}  z_mean = {y_val.mean():.4f}  z_std = {y_val.std():.4f}")
print(f"Test  : n = {len(y_test):>7,}  z_mean = {y_test.mean():.4f}  z_std = {y_test.std():.4f}")

print(f"\nScalars (log10, brutos) — range no train:")
for i, name in enumerate(SCALARS_CONFIG_FEATURES):
    s = S_train[:, i]
    print(f"  {name:>12} : min={s.min():.3f}  median={np.median(s):.3f}  max={s.max():.3f}")


In [ ]:
# ============================================================
# PLOT DAS DISTRIBUICOES — sanity check da estratificacao + scalars
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))

# (a) histograma de z
bins = np.linspace(float(y_all.min()), float(y_all.max()), 50)
for arr, label in [(y_train, "train"), (y_val, "val"), (y_test, "test")]:
    axes[0].hist(arr, bins=bins, density=True, alpha=0.5, label=f"{label} (n={len(arr):,})")
axes[0].set_xlabel("z"); axes[0].set_ylabel("densidade")
axes[0].set_title("(a) Distribuicao de z (deve ficar parecida nos 3 splits)")
axes[0].legend()

# (b) CDF de z
for arr, label in [(y_train, "train"), (y_val, "val"), (y_test, "test")]:
    s = np.sort(arr)
    axes[1].plot(s, np.arange(1, len(s)+1)/len(s), label=label, lw=2)
axes[1].set_xlabel("z"); axes[1].set_ylabel("CDF")
axes[1].set_title("(b) CDF de z — splits estratificados devem se sobrepor")
axes[1].legend()

# (c) scalars (log10) no TRAIN — valores fisicamente interpretaveis
axes[2].boxplot(
    [S_train[:, i] for i in range(n_scalars)],
    tick_labels=SCALARS_CONFIG_FEATURES, showfliers=False,
)
axes[2].set_ylabel("log10 (valor)")
axes[2].set_title("(c) Scalars no TRAIN (log10 puro, sem z-score)")
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()


## 5. Helpers — metricas e pesos por z


In [ ]:
# ============================================================
# METRICAS DE PHOTO-Z / SPEC-Z (convencao da literatura)
# ============================================================
# Todas operam sobre delta_z = (z_pred - z_true) / (1 + z_true), o ERRO
# NORMALIZADO. Dividir por (1+z) compensa o efeito da dilatacao temporal
# cosmologica — assim erros sao comparaveis entre z baixo e z alto.

def delta_z_norm(z_pred, z_true):
    """Erro normalizado. Padronizado em photo-z desde ~2006 (DECaLS, KiDS)."""
    return (z_pred - z_true) / (1.0 + z_true)


def bias(dz):
    """Vies sistematico = mediana de delta_z.

    Idealmente = 0. Se positivo, modelo SUPER-prediz z (acha que galaxia esta
    mais longe que esta). Se negativo, SUB-prediz. Mediana (nao media!) pra
    ser robusto a outliers catastroficos.
    """
    return float(np.median(dz))


def sigma_NMAD(dz):
    """Dispersao robusta = 1.4826 * MAD(delta_z).
b
    MAD = mediana(|delta_z - mediana(delta_z)|). Constante 1.4826 vem de
    Tukey: pra uma distribuicao normal, sigma_NMAD == desvio padrao usual.
    Diferenca: NMAD ignora caudas pesadas (outliers nao inflam o numero).
    """
    return float(1.4826 * np.median(np.abs(dz - np.median(dz))))


def eta_outliers(dz, thr=0.15):
    """% de catastroficos (|delta_z| > thr).

    thr=0.15 e convencao da literatura photo-z (Euclid, DES, KiDS). Em
    spec-z (precisao maior), eta_outliers a 0.15 fica <0.5% — talvez valha
    olhar tambem a thr=0.05 pra discriminar entre modelos.
    """
    return float(100.0 * np.mean(np.abs(dz) > thr))


def f_inliers(dz, thr=0.15):
    """% de bons (|delta_z| <= thr). Complemento de eta_outliers."""
    return float(100.0 * np.mean(np.abs(dz) <= thr))

def composite_objective(dz, w_sigma, sigma_ref, eta_ref, eps=1e-9):
    """score = w * (sigma_NMAD + |bias|)/sigma_ref + (1-w) * eta_outliers/eta_ref."""
    s = sigma_NMAD(dz)
    b = abs(bias(dz))
    e = eta_outliers(dz)
    return w_sigma * (s + b) / (sigma_ref + eps) + (1 - w_sigma) * e / (eta_ref + eps)


def make_redshift_weights(y, z_focus_low, z_focus_high,
                          w_low, w_focus, w_high, w_max, enabled):
    if not enabled:
        return None
    w = np.full_like(y, w_low, dtype=np.float32)
    w = np.where(y < z_focus_low,                       w_low,   w)
    w = np.where((y >= z_focus_low) & (y <= z_focus_high), w_focus, w)
    w = np.where(y > z_focus_high,                      w_high,  w)
    return np.minimum(w, w_max)


## 6. Refs do composite objective (baseline trivial)


In [ ]:
# ============================================================
# REFERENCIAS DO COMPOSITE OBJECTIVE — baseline trivial
# ============================================================
# So usado se OBJECTIVE_MODE == "composite" (default agora e "mae", entao
# isso aqui fica de "escala de referencia").
#
# Ideia: pra normalizar o composite_objective, precisamos saber "quao mal
# erra um modelo BURRO?". Modelo burro = "sempre preve a mediana de y" —
# nao olha o espectro, so chuta o valor central. Calculamos sigma_NMAD e
# eta_outliers desse modelo trivial e usamos como divisor no composite.
#
# Score = 1 -> tao bom quanto chutar mediana (ruim)
# Score < 1 -> melhor que mediana
# Score ~ 0.1 -> 10x melhor que mediana

y_med = float(np.median(y_pool))                                # mediana do train+val
dz_ref = delta_z_norm(np.full_like(y_pool, y_med), y_pool)      # erro do "burrao"

sigma_NMAD_ref   = sigma_NMAD(dz_ref)
eta_outliers_ref = max(eta_outliers(dz_ref), 1.0)               # max(., 1) evita div/0 se 0% de outliers

print(f"Baseline trivial (prever sempre mediana = {y_med:.4f}):")
print(f"  sigma_NMAD_ref   = {sigma_NMAD_ref:.4f}    (dispersao do erro do burrao)")
print(f"  eta_outliers_ref = {eta_outliers_ref:.2f}%   (% de catastroficos do burrao)")


## 7. `build_cnn_from_trial` — single input (so o espectro)

### Estrutura

```
spectrum (n_wave, 1) ───> Conv blocks ───> GAP ───> Dense head ──> z
```

- O **espectro normalizado** (`X_train`/`X_val`/`X_test`) e o UNICO input.
- Os scalars sao computados e salvos paralelamente, mas **nao entram** na rede.
- Saida: `Dense(1) + ScaledSoftplus` (z >= 0).

### Por que single-input?
Decisao cientifica: spec-z deve depender da FORMA do espectro (linhas, breaks, continuum shape relativo), nao do brilho absoluto. Ver intro do notebook pros 4 motivos.

### HPs e detalhes
HPs do Optuna, dropout escalonado, Conv1D+BN+MaxPool, AdamW + clipnorm — sem mudancas.



In [ ]:
def build_cnn_from_trial(trial, n_wave: int, seed: int = 42):
    """CNN single-input: so o espectro normalizado.

    Retorna (modelo, batch_size). Modelo aceita X (n_samples, n_wave, 1).
    Scalars sao calculados/salvos paralelamente mas NAO entram no modelo
    (decisao cientifica — ver intro do notebook).
    """

    # ============================================================
    # HIPERPARAMETROS QUE O OPTUNA ESCOLHE
    # ============================================================
    # trial.suggest_int/float/categorical: cada chamada PEDE ao Optuna um valor
    # dentro do range definido em HP_SPACE. O Optuna usa TPE pra "chutar
    # esperto" baseado nos trials anteriores.
    #
    # IMPORTANTE: a ORDEM e os NOMES dessas chamadas viram a "assinatura" do
    # trial — sao usadas pra calcular best_params e replicar arquitetura no
    # retreino final (via FixedTrial).
    n_conv_blocks  = trial.suggest_int("n_conv_blocks",
                                       HP_SPACE["n_conv_blocks"]["low"],
                                       HP_SPACE["n_conv_blocks"]["high"])
    n_dense_layers = trial.suggest_int("n_dense_layers",
                                       HP_SPACE["n_dense_layers"]["low"],
                                       HP_SPACE["n_dense_layers"]["high"])
    n_units        = trial.suggest_int("n_units",
                                       HP_SPACE["n_units"]["low"],
                                       HP_SPACE["n_units"]["high"],
                                       step=HP_SPACE["n_units"]["step"])
    kernel_size    = trial.suggest_categorical("kernel_size", HP_SPACE["kernel_size"])
    dropout_rate   = trial.suggest_float("dropout_rate",
                                         HP_SPACE["dropout_rate"]["low"],
                                         HP_SPACE["dropout_rate"]["high"])
    learning_rate  = trial.suggest_float("learning_rate",
                                         HP_SPACE["learning_rate"]["low"],
                                         HP_SPACE["learning_rate"]["high"],
                                         log=HP_SPACE["learning_rate"]["log"])
    weight_decay   = trial.suggest_float("weight_decay",
                                         HP_SPACE["weight_decay"]["low"],
                                         HP_SPACE["weight_decay"]["high"],
                                         log=HP_SPACE["weight_decay"]["log"])
    activation     = trial.suggest_categorical("activation", HP_SPACE["activation"])
    batch_size     = trial.suggest_categorical("batch_size", HP_SPACE["batch_size"])
    use_batchnorm  = trial.suggest_categorical("use_batchnorm", HP_SPACE["use_batchnorm"])
    dropout_schedule = trial.suggest_categorical("dropout_schedule", HP_SPACE["dropout_schedule"])
    optimizer_name   = trial.suggest_categorical("optimizer", HP_SPACE["optimizer"])
    taper_last_block = trial.suggest_categorical("taper_last_block", HP_SPACE["taper_last_block"])


    # ============================================================
    # INPUTS — espectro (CNN) + scalars (concatenados depois)
    # ============================================================
    # GlorotUniform com seed fixa: inicializacao reprodutivel (sem isso, dois
    # runs com mesmos HPs dao resultados ligeiramente diferentes).
    init = keras.initializers.GlorotUniform(seed=seed)

    # Single input: so o espectro normalizado entra no modelo. Os scalars sao
    # calculados/salvos pra analise (ver cell de extract_scalars), mas NAO entram
    # aqui — decisao cientifica: spec-z deve depender da FORMA do espectro, nao
    # do brilho absoluto (que depende de calibracao instrumental + selection
    # effects do eBOSS, podendo nao generalizar pra outras surveys).
    spec_input = keras.Input(shape=(n_wave, 1), name="spectrum")


    # ============================================================
    # PARTE CONVOLUCIONAL — extrai features da FORMA do espectro
    # ============================================================
    # Cada bloco: Conv1D -> BN -> Conv1D -> BN -> MaxPool(2) -> Dropout
    #
    # Por que 2 Conv1D em sequencia (sem MaxPool no meio)?
    #   Aumenta o "campo receptivo" do bloco (~2x kernel_size) sem reduzir
    #   a resolucao espacial cedo. So depois o MaxPool corta pela metade.
    #
    # Por que BN entre cada Conv?
    #   BatchNormalization estabiliza o treino (normaliza ativacoes), age
    #   como leve regularizador, e permite usar learning rate maior.
    #
    # Estrategia de filtros: 64, 128, 256, 256 (capado em 256 pra economizar
    # VRAM em GPUs menores). Mais profundo = mais filtros, padrao classico.
    #
    # Estrategia de kernel: comeca em kernel_size, reduz 6 por bloco (piso 3).
    # Bloco 1 ve features largas (continuum, breaks); blocos profundos veem
    # features finas (linhas estreitas).
    #
    # Dropout escalonado: comeca leve (0.5*dropout_rate) no bloco 1 e cresce
    # ate pleno (1.0*dropout_rate) no ultimo bloco. Comum em CNNs profundas.
    x = spec_input
    for b in range(n_conv_blocks):
        k = max(3, kernel_size - 6 * b)                # kernel diminui por bloco

        # BLOCO DE AFUNILAMENTO (estilo baseline): so no ULTIMO bloco, se ligado.
        # 1 conv, filtros pela METADE, SEM MaxPool e SEM Dropout -> vai direto pro
        # GlobalAveragePooling. Reproduz o 4o bloco do baseline (64,128,256,128).
        # Sem isso o flex so consegue 64,128,256,256 (2 conv + pool sempre).
        if taper_last_block and b == n_conv_blocks - 1:
            n_filters = min(64 * (2 ** b), 256) // 2   # ex.: 256 -> 128
            x = keras.layers.Conv1D(n_filters, k, activation=activation,
                                    padding="same", kernel_initializer=init)(x)
            if use_batchnorm:
                x = keras.layers.BatchNormalization()(x)
            continue

        n_filters = min(64 * (2 ** b), 256)            # 64, 128, 256, 256
        if dropout_schedule == "ramp":
            dr = dropout_rate * (0.5 + 0.5 * (b + 1) / n_conv_blocks)
        else:  # flat — leve e constante (estilo baseline)
            dr = dropout_rate

        x = keras.layers.Conv1D(n_filters, k, activation=activation,
                                padding="same", kernel_initializer=init)(x)
        if use_batchnorm:
            x = keras.layers.BatchNormalization()(x)
        x = keras.layers.Conv1D(n_filters, k, activation=activation,
                                padding="same", kernel_initializer=init)(x)
        if use_batchnorm:
            x = keras.layers.BatchNormalization()(x)
        x = keras.layers.MaxPooling1D(2)(x)
        x = keras.layers.Dropout(dr, seed=seed)(x)

    # GlobalAveragePooling1D: mata a dimensao espacial (de N x L x F vira N x F).
    # Equivale a "media ao longo do comprimento de onda" pra cada filtro.
    # Mais robusto que Flatten (que explode o numero de parametros) e
    # naturalmente invariante a translacoes pequenas no espectro.
    x = keras.layers.GlobalAveragePooling1D()(x)


    # ============================================================
    # DENSE HEAD — processa o vetor de features e prediz z
    # ============================================================
    # Funil classico: cada camada corta units pela metade (piso 64).
    # Dropout decresce conforme se aproxima da saida — perto do output,
    # menos regularizacao (caso contrario o sinal final fica ruidoso demais).
    units = n_units
    for d in range(n_dense_layers):
        x = keras.layers.Dense(units, activation=activation, kernel_initializer=init)(x)
        x = keras.layers.BatchNormalization()(x)
        dr_dense = (dropout_rate * max(0.4, 1.0 - 0.2 * d)
                    if dropout_schedule == "ramp" else dropout_rate)
        x = keras.layers.Dropout(dr_dense, seed=seed)(x)
        units = max(64, units // 2)


    # ============================================================
    # OUTPUT — Dense(1) + ScaledSoftplus (z >= 0, sem teto)
    # ============================================================
    # Dense(1) com saida linear (sem activation aqui!) -> pode dar qualquer real.
    # ScaledSoftplus = softplus(beta*x)/beta, garante saida >= 0 (z fisicamente
    # so pode ser positivo). beta=10 controla o "sharpness" perto de zero.
    x = keras.layers.Dense(1)(x)
    out = ScaledSoftplus(beta=10.0, name="redshift")(x)


    # ============================================================
    # COMPILAR — AdamW + MSE
    # ============================================================
    # AdamW (em vez de Adam puro): aplica weight_decay corretamente como
    # regularizacao L2 "decoupled" — pesos sao reduzidos diretamente em vez
    # de via gradiente. Mais eficaz que Adam + L2 manual.
    #
    # clipnorm=1.0: corta gradientes com norma > 1, evita explosao em
    # learning rates altos (sem isso, AdamW pode dar NaN).
    #
    # loss="mse": Mean Squared Error pro TREINO. Gradiente suave perto do
    # otimo (vs MAE com gradiente constante).
    # metrics=["mae"]: so reporta, nao afeta o treino. MAE eh mais
    # interpretavel pra acompanhar a curva.
    model = keras.Model(inputs=spec_input, outputs=out,
                        name="CNN_flex_optuna")

    # optimizer e um LEVER DE PIPELINE: "adamw" = receita do flex (weight decay
    # desacoplado); "adam" = receita do baseline manual (sem weight decay).
    # Deixa o Optuna decidir qual pipeline generaliza melhor.
    if optimizer_name == "adamw":
        opt = keras.optimizers.AdamW(
            learning_rate=learning_rate,
            weight_decay=weight_decay,
            beta_1=0.9, beta_2=0.999,
            clipnorm=1.0,
        )
    else:  # adam puro — igual ao baseline (weight_decay nao se aplica)
        opt = keras.optimizers.Adam(
            learning_rate=learning_rate,
            beta_1=0.9, beta_2=0.999,
            clipnorm=1.0,
        )
    model.compile(optimizer=opt, loss="mse", metrics=["mae"])

    return model, batch_size

## 8. Pruning callback do Optuna


In [ ]:
class OptunaPruningCallback(keras.callbacks.Callback):
    """Conecta o treino do Keras ao Pruner do Optuna.

    Keras Callbacks tem varios "hooks" que sao chamados em momentos especificos
    do treino (on_train_begin, on_epoch_end, on_batch_end, etc.). A gente
    override so o on_epoch_end pra reportar val_loss a cada fim de epoca.

    Fluxo:
      1. Keras chama on_epoch_end(epoch, logs) ao fim de cada epoca.
      2. logs dict contem as metricas: {"loss": ..., "val_loss": ..., "val_mae": ...}.
      3. A gente reporta o val_loss ao Optuna via trial.report.
      4. Pergunta ao Optuna se este trial deve ser podado.
      5. Se sim, levanta TrialPruned (Optuna intercepta no study.optimize).
    """
    def __init__(self, trial, metric="val_loss"):
        super().__init__()              # IMPORTANTE: sempre chamar super().__init__ em Callback
        self.trial  = trial
        self.metric = metric

    def on_epoch_end(self, epoch, logs=None):
        # `(logs or {}).get(...)`: se logs for None, usa dict vazio. .get() retorna
        # None se a chave nao existe (mais seguro que logs[self.metric] que daria KeyError).
        v = (logs or {}).get(self.metric)
        if v is None:
            return                       # se nao tem val_loss neste log, nao faz nada

        # Reporta este valor ao Optuna (e o step = epoca atual)
        self.trial.report(float(v), step=epoch)

        # Pergunta ao Optuna se vale a pena continuar este trial
        if self.trial.should_prune():
            # raise pra Optuna capturar e marcar trial como PRUNED
            raise optuna.TrialPruned(f"epoch={epoch} {self.metric}={v:.4f}")


## 9. `objective` — treina e devolve score

Cada chamada:

1. Limpa sessao Keras.
2. Constroi CNN single-input (so espectro).
3. Treina passando `X_train` direto (sem dict — scalars nao entram no modelo).
4. Calcula score (MAE ou composite) no val.
5. Devolve o score — Optuna minimiza.



In [ ]:
def objective(trial):
    """Treina CNN com HPs do trial; devolve score no val."""

    # ============================================================
    # 1. LIMPA SESSAO KERAS + REPRODUTIBILIDADE
    # ============================================================
    # keras.backend.clear_session(): destroi o grafo TF da memoria. Sem isso,
    # cada trial novo ADICIONA mais grafo a memoria da GPU -> OOM em ~10-20 trials.
    tf.keras.backend.clear_session()

    # Seed diferente por trial (SEED + trial.number): garante reprodutibilidade
    # MAS evita que todos os trials comecem com inicializacao identica
    # (importante quando dois trials tem os mesmos HPs por acaso).
    set_reproducibility(SEED + trial.number)


    # ============================================================
    # 2. CONSTROI A CNN COM HPS SORTEADOS PELO TRIAL
    # ============================================================
    # build_cnn_from_trial chama trial.suggest_* internamente — isso REGISTRA
    # no Optuna quais HPs foram escolhidos pra este trial.
    model, batch_size = build_cnn_from_trial(
        trial, n_wave=n_wave, seed=SEED + trial.number,
    )


    # ============================================================
    # 3. SAMPLE WEIGHTS POR Z (opcional, ligado em WEIGHTS_CONFIG)
    # ============================================================
    # Se use_z_weights=True, certas faixas de z ganham peso maior no calculo
    # da loss. Util pra "ensinar" o modelo a focar em z dificeis de aprender.
    # Por default (False), sw_train=None -> Keras usa peso 1.0 pra tudo.
    sw_train = make_redshift_weights(
        y_train,
        **{k: WEIGHTS_CONFIG[k] for k in
            ["z_focus_low", "z_focus_high", "w_low", "w_focus", "w_high", "w_max"]},
        enabled=WEIGHTS_CONFIG["use_z_weights"],
    )


    # ============================================================
    # 4. CALLBACKS — pruning + early stopping + reduce LR
    # ============================================================
    # OptunaPruningCallback: reporta val_loss ao Optuna a cada epoca.
    #   Se este trial esta entre os piores (pelo percentile do pruner), e
    #   morto cedo via TrialPruned. Economiza compute.
    #
    # EarlyStopping: para o treino deste trial se val_loss nao melhorar
    #   em N epocas seguidas. restore_best_weights traz os pesos da
    #   melhor epoca (em vez dos da ultima, possivelmente ja em overfit).
    #
    # ReduceLROnPlateau: se val_loss estagnar, reduz lr pela metade. Ajuda
    #   no "fine tuning" final, quando o modelo precisa passos menores.
    cbs = [
        OptunaPruningCallback(trial, metric="val_loss"),
        keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=TRAINING_CONFIG["early_stop_patience"],
            restore_best_weights=True,
            min_delta=TRAINING_CONFIG["early_stop_min_delta"],
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=TRAINING_CONFIG["reduce_lr_factor"],
            patience=TRAINING_CONFIG["reduce_lr_patience"],
            min_lr=TRAINING_CONFIG["reduce_lr_min"],
        ),
    ]


    # ============================================================
    # 5. TREINA — try/except deixa o pruner levantar TrialPruned
    # ============================================================
    # Se o callback decidir podar, levanta optuna.TrialPruned -> a gente
    # PROPAGA pra o Optuna capturar e marcar trial como PRUNED.
    # Outras excecoes (ex: OOM) sao engolidas la em cima (study.optimize
    # com catch=ResourceExhaustedError).
    try:
        history = model.fit(
            X_train, y_train,
            validation_data=(X_val, y_val),
            epochs=EPOCHS_PER_TRIAL,
            batch_size=batch_size,
            sample_weight=sw_train,
            callbacks=cbs,
            verbose=0,
        )
    except optuna.TrialPruned:
        raise


    # ============================================================
    # 6. SCORE NO VAL — usado pelo Optuna pra ranquear trials
    # ============================================================
    # Predicao no val (so val, NUNCA test — test fica intocado ate o fim).
    y_val_pred = model.predict(X_val, verbose=0).ravel()
    dz_val = delta_z_norm(y_val_pred, y_val)

    # Dois modos disponiveis:
    #   "mae"      -> score = MAE direto (simples, robusto, default agora)
    #   "composite"-> w*(sigma + |bias|)/sigma_ref + (1-w)*outliers/outlier_ref
    #                 (heranca do photo-z; em spec-z outliers ja sao raros).
    if OBJECTIVE_CONFIG["mode"] == "mae":
        score = float(np.mean(np.abs(y_val_pred - y_val)))
    else:
        score = composite_objective(
            dz_val,
            w_sigma   = OBJECTIVE_CONFIG["w_sigma"],
            sigma_ref = sigma_NMAD_ref,
            eta_ref   = eta_outliers_ref,
        )


    # ============================================================
    # 7. EXTRAS — guarda no trial pra inspecao depois
    # ============================================================
    # set_user_attr grava info adicional no DB do Optuna. Aparece em
    # study.trials e na CSV do top_trials. Util pra entender POR QUE
    # um trial foi bom (ex: "achou epoch otima cedo" vs "treinou ate o fim").
    trial.set_user_attr("best_epoch", int(np.argmin(history.history["val_loss"]) + 1))
    trial.set_user_attr("val_mae",    float(np.mean(np.abs(y_val_pred - y_val))))
    trial.set_user_attr("val_sigma",  float(sigma_NMAD(dz_val)))
    trial.set_user_attr("val_bias",   float(bias(dz_val)))
    trial.set_user_attr("val_eta_15", float(eta_outliers(dz_val, 0.15)))

    return score

## 10. RUN_TAG + diretorios + storage SQLite


In [ ]:
# ============================================================
# RUN_TAG — id legivel do experimento
# ============================================================
# Concatena varios pedacos em uma string descritiva:
#   "flex-LRG-t50_e40_s42-mae_nozw-abc12345"
#         |    |          |         |
#         |    t = trials  objetivo  hash da config
#         |
#         OBJECT_TYPE
#
# String concatenation em Python:
#   - f-strings adjacentes sao concatenadas implicitamente (1a e 2a linhas)
#   - "+" eh concatenacao explicita (3a linha)
#   - ("_zw" if USE_Z_WEIGHTS else "_nozw"): conditional expression
#     (ternario: "valor1 if condicao else valor2")
RUN_TAG = (
    f"flex-{OBJECT_TYPE}-t{N_TRIALS}_e{EPOCHS_PER_TRIAL}_s{SEED}"
    f"-{OBJECTIVE_MODE}"
    + ("_zw" if USE_Z_WEIGHTS else "_nozw")
    + f"-{config_hash}"
)


# ============================================================
# DIRETORIOS — pathlib em vez de strings
# ============================================================
# Path / "subpasta": operator overload do pathlib. Equivale a os.path.join
# mas e mais legivel e funciona cross-platform (linux usa / mas windows usa \).
BASE_RESULTS  = RESULTS_DIR / OBJECT_TYPE / "cnn_optuna_flex"
RUN_DIR       = BASE_RESULTS / RUN_TAG
PLOT_DIR      = RUN_DIR / "plots"
MODEL_RUN_DIR = MODELS_DIR / OBJECT_TYPE / "cnn_optuna_flex" / RUN_TAG

# Cria todos os diretorios necessarios.
#   parents=True: cria diretorios intermediarios se nao existirem (tipo mkdir -p)
#   exist_ok=True: nao levanta erro se ja existe (idempotente — pode rodar de novo)
for d in (BASE_RESULTS, RUN_DIR, PLOT_DIR, MODEL_RUN_DIR):
    d.mkdir(parents=True, exist_ok=True)


# ============================================================
# SQLite STORAGE — sobrevive a quedas via load_if_exists=True
# ============================================================
# Optuna armazena trials num banco SQLite (arquivo .db). Vantagens:
#   - Sobrevive se o job cair (re-submeter continua de onde parou)
#   - Multiplos jobs podem rodar e ver o mesmo study (parallelizacao)
#   - Inspecionavel: optuna-dashboard, dbeaver, ou sqlite3 direto
DB_PATH     = BASE_RESULTS / "optuna_study.db"
STORAGE_URL = f"sqlite:///{DB_PATH}"     # formato URI esperado pelo Optuna

# study_name inclui config_hash: se voce mudar HP_SPACE ou outro config,
# o hash muda -> Optuna cria study NOVO no mesmo .db (nao mistura experimentos).
STUDY_NAME  = f"cnn_flex_{OBJECT_TYPE}_{OBJECTIVE_MODE}_{config_hash}"

print(f"RUN_TAG    : {RUN_TAG}")
print(f"RUN_DIR    : {RUN_DIR}")
print(f"MODEL_DIR  : {MODEL_RUN_DIR}")
print(f"STORAGE    : {STORAGE_URL}")
print(f"STUDY_NAME : {STUDY_NAME}")


In [ ]:
study = optuna.create_study(
    study_name=STUDY_NAME,
    storage=STORAGE_URL,
    load_if_exists=True,
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=SEED),
    pruner=optuna.pruners.PercentilePruner(
        percentile=HPO_CONFIG["pruner_percentile"],
        n_startup_trials=HPO_CONFIG["n_startup_trials"],
        n_warmup_steps=HPO_CONFIG["n_warmup_steps"],
        interval_steps=HPO_CONFIG["interval_steps"],
    ),
)

# --- FIX (2026-06-10): semeia o BASELINE (informado por dominio) como trial inicial ---
# Garante que o Optuna AVALIE a arquitetura que ja sabemos boa -> o melhor trial
# nao fica abaixo do baseline manual. skip_if_exists evita duplicar no resume.
_baseline_hp = {
    "n_conv_blocks": 4, "n_dense_layers": 3, "n_units": 256, "kernel_size": 21,
    "dropout_rate": 0.20, "learning_rate": 3e-4, "weight_decay": 1e-4,
    "activation": "relu", "batch_size": 64,
    "use_batchnorm": True,                             # baseline USA BatchNorm (faltava -> trial 0 saía sem BN!)
    "dropout_schedule": "flat", "optimizer": "adam",
    "taper_last_block": True,                          # baseline afunila o 4o bloco (64,128,256,128)
}
try:
    study.enqueue_trial(_baseline_hp, skip_if_exists=True)
except TypeError:  # optuna antigo sem skip_if_exists
    if sum(t.state != optuna.trial.TrialState.WAITING for t in study.trials) == 0:
        study.enqueue_trial(_baseline_hp)
print("Baseline (dominio) enfileirado como trial inicial.")


n_complete = sum(t.state == optuna.trial.TrialState.COMPLETE for t in study.trials)
n_pruned   = sum(t.state == optuna.trial.TrialState.PRUNED   for t in study.trials)
print(f"Study : {study.study_name}")
print(f"Trials: {len(study.trials)}  ({n_complete} completos | {n_pruned} podados)")


## 11. Rodar o Optuna


In [ ]:
# ============================================================
# CALLBACK DE LOG — chamado pelo Optuna apos cada trial
# ============================================================
# Optuna chama essa funcao com (study, trial) ao fim de cada trial.
# Imprime uma linha resumindo o trial (numero, estado, score, extras).
def trial_logger(study, trial):
    # trial.value: score do trial (None se PRUNED ou FAIL)
    s = trial.value if trial.value is not None else float("nan")

    # Acumula extras numa lista, junta com " | " no print
    extras = []
    if trial.state == optuna.trial.TrialState.COMPLETE:
        # .get(key, default): retorna default se key nao existe (em vez de KeyError)
        extras.append(f"epoch={trial.user_attrs.get('best_epoch', '-')}")
        extras.append(f"sigma={trial.user_attrs.get('val_sigma', float('nan')):.4f}")

    # " | ".join(lista): junta strings com separador. Se lista vazia, retorna ""
    extras_str = " | ".join(extras) if extras else ""

    # f-string formato:
    #   {trial.number:3d}    - inteiro em 3 chars com padding (ex: "  5", " 50", "500")
    #   {trial.state.name:>10} - string alinhada a direita em 10 chars
    #   {s:.5f}              - float com 5 casas decimais
    # flush=True: forca print imediato (sem isso, pode bufferizar em jobs longos)
    print(f"  trial {trial.number:3d} state={trial.state.name:>10} score={s:.5f}  {extras_str}",
          flush=True)


# ============================================================
# LOOP DE OPTIMIZE — ate atingir N_TRIALS COMPLETOS
# ============================================================
# Por que while em vez de so chamar study.optimize(n_trials=N)?
#   - PRUNED e FAIL nao contam como "completos"
#   - Se queremos N completos especificamente, precisamos re-checar e
#     pedir mais ate atingir
#   - Tambem ajuda se o job cair no meio: ao re-submeter, ja tem
#     trials no DB e o while pede so o que falta
target = TARGET_TRIALS_OVERRIDE if TARGET_TRIALS_OVERRIDE else HPO_CONFIG["target_trials"]

# Conta trials COMPLETOS atuais. Sum sobre generator: cada True conta 1.
n_complete = sum(t.state == optuna.trial.TrialState.COMPLETE for t in study.trials)

while n_complete < target:
    n_remaining = target - n_complete
    print(f"\n>>> rodando {n_remaining} trials adicionais...", flush=True)
    study.optimize(
        objective,                                    # funcao do trial (define cima)
        n_trials=n_remaining,                         # rodar so o que falta
        callbacks=[trial_logger],                     # lista de callbacks (poderia ter mais)
        gc_after_trial=True,                          # roda garbage collector apos cada trial (libera memoria)
        show_progress_bar=False,                      # progress bar polui log do papermill
        # catch: tupla de exceptions a engolir. Se OOM acontecer, trial vira
        # FAIL no DB e o study continua. Sem isso, OOM mata o job inteiro.
        catch=(tf.errors.ResourceExhaustedError,),
    )
    # Recalcula apos cada batch de trials. Loop sai quando n_complete >= target.
    n_complete = sum(t.state == optuna.trial.TrialState.COMPLETE for t in study.trials)


# ============================================================
# RELATORIO FINAL DO HPO
# ============================================================
n_pruned = sum(t.state == optuna.trial.TrialState.PRUNED for t in study.trials)
print(f"\nHPO concluido: {n_complete} completos | {n_pruned} podados")
print(f"\nBest value : {study.best_value:.5f}  ({OBJECTIVE_MODE})")
print(f"Best trial : #{study.best_trial.number}")
print(f"Best params:")
# .items() devolve pares (key, value). {k:>16}: chave alinhada a direita em 16 chars.
for k, v in study.best_trial.params.items():
    print(f"  {k:>16} = {v}")


## 12. Top trials + plots de diagnostico


In [ ]:
# ============================================================
# CONSTROI TABELA DOS TRIALS COMPLETOS — itera + extrai info
# ============================================================
# Estrategia: lista de dicts -> DataFrame. Cada dict eh uma linha.
rows = []
for t in study.trials:
    # Pula trials PRUNED ou FAIL
    if t.state != optuna.trial.TrialState.COMPLETE:
        continue

    # Cria dict com info do trial. **t.params eh "unpacking" — espalha as chaves
    # do dict t.params dentro deste dict. Equivale a:
    #   for k, v in t.params.items():
    #       row[k] = v
    # Mas mais conciso. Resulta em colunas pros HPs (n_conv_blocks, lr, etc).
    rows.append({
        "trial":      t.number,
        "score":      t.value,
        "best_epoch": t.user_attrs.get("best_epoch"),
        "val_mae":    t.user_attrs.get("val_mae"),
        "val_sigma":  t.user_attrs.get("val_sigma"),
        "val_bias":   t.user_attrs.get("val_bias"),
        "val_eta_15": t.user_attrs.get("val_eta_15"),
        **t.params,                                   # <-- aqui o unpacking
    })

# pd.DataFrame(list_of_dicts): cria DataFrame onde cada dict eh uma linha
# e as chaves viram colunas. Colunas que faltam em algum dict viram NaN.
# .sort_values("score"): ordena ascending por padrao (menores scores primeiro = melhores)
df_top = pd.DataFrame(rows).sort_values("score")

# Salva CSV. index=False: nao escreve a coluna de indice (0, 1, 2...).
df_top.to_csv(RUN_DIR / "optuna_top.csv", index=False)
print(f"Salvo: {RUN_DIR / 'optuna_top.csv'}")

# Mostra os top 10 com 5 casas decimais (ultima expressao da cell = saida do jupyter)
df_top.head(10).round(5)


In [ ]:
# ============================================================
# 3 PLOTS DIAGNOSTICOS DO OPTUNA
# ============================================================
# plt.subplots(1, 3, ...): figura com 1 linha x 3 colunas de plots.
# Devolve (fig, axes) onde axes eh um array de 3 axes (axes[0], axes[1], axes[2]).
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))

# Extrai os valores das colunas como arrays numpy (mais rapido que series pandas)
trials_x = df_top["trial"].values
trials_y = df_top["score"].values

# Ordena por numero do trial (df_top esta ordenado por score, queremos por trial)
order = np.argsort(trials_x)                          # indices que ordenam trials_x
xs, ys = trials_x[order], trials_y[order]             # aplica os indices em ambos

# np.minimum.accumulate: para cada posicao i, calcula min(ys[0..i]).
# Ex: ys=[5, 3, 7, 2, 8] -> [5, 3, 3, 2, 2] = "melhor ate aqui"
# Curva monotonicamente decrescente.
best_so_far = np.minimum.accumulate(ys) if len(ys) else ys


# --- (a) Convergencia do Optuna ---
axes[0].plot(xs, ys, "o", alpha=0.5, color="steelblue", label="trial")   # bolinhas = cada trial
axes[0].plot(xs, best_so_far, "-", lw=2, color="green", label="best so far")  # linha verde = melhor ate ali

# List comprehension: cria lista de numeros de trials PODADOS
pruned_x = [t.number for t in study.trials if t.state == optuna.trial.TrialState.PRUNED]
if pruned_x and len(ys):
    # Marca podados no topo do plot. ys.max() * len() cria uma lista de um valor repetido.
    axes[0].scatter(pruned_x, [ys.max()] * len(pruned_x),
                    marker="x", color="gray", alpha=0.5, label="pruned")
axes[0].set_xlabel("Trial"); axes[0].set_ylabel(f"score ({OBJECTIVE_MODE})")
axes[0].set_title("(a) Convergencia do Optuna"); axes[0].legend()
axes[0].set_yscale("log")                             # log scale: melhora visualizacao se range eh grande


# --- (b) Histograma dos scores ---
axes[1].hist(ys, bins=20, color="steelblue", alpha=0.8, edgecolor="black", lw=0.3)
# axvline: linha vertical no eixo x. Marca o melhor score.
axes[1].axvline(study.best_value, color="red", lw=2, label=f"melhor: {study.best_value:.5f}")
axes[1].set_xlabel(f"score ({OBJECTIVE_MODE})"); axes[1].set_ylabel("# trials")
axes[1].set_title("(b) Distribuicao dos scores"); axes[1].legend()


# --- (c) Importancia dos HPs (pode falhar se nao houver trials completos suficientes) ---
# try/except: se Optuna nao conseguir computar importancias (ex: poucos trials),
# mostra mensagem no plot em vez de quebrar.
try:
    # optuna.importance.get_param_importances: estima importancia via fANOVA
    importances = optuna.importance.get_param_importances(study)
    # barh = horizontal bar chart. Mais legivel pra nomes longos de HPs.
    axes[2].barh(list(importances.keys()), list(importances.values()),
                 color="coral", edgecolor="black", lw=0.5)
    axes[2].set_xlabel("Importancia relativa")
    axes[2].set_title("(c) Importancia dos hiperparametros")
    axes[2].invert_yaxis()                            # mais importante no topo
except Exception as e:
    # type(e).__name__ pega o nome da classe da excecao (ex: "RuntimeError")
    # transform=axes[2].transAxes: coords em fracao do plot (0.5, 0.5 = centro)
    axes[2].text(0.5, 0.5, f"Importances indisponivel:\n{type(e).__name__}",
                 ha="center", va="center", transform=axes[2].transAxes)
    axes[2].set_title("(c) Importancia dos hiperparametros")


plt.tight_layout()                                    # ajusta espacamento entre subplots
plt.savefig(PLOT_DIR / "optuna_summary.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Salvo: {PLOT_DIR / 'optuna_summary.png'}")


## 13. Retreino final com os melhores hiperparametros


In [ ]:
class TrainInferenceMAECallback(keras.callbacks.Callback):
    """MAE no train em modo inferencia (dropout off)."""
    def __init__(self, x_tr, y_tr, max_eval_samples=None, seed=42):
        super().__init__()
        n = len(y_tr)
        if max_eval_samples is not None and max_eval_samples < n:
            rng = np.random.default_rng(seed)
            idx = rng.choice(n, size=max_eval_samples, replace=False)
            self.x_tr = x_tr[idx]
            self.y_tr = y_tr[idx]
        else:
            self.x_tr = x_tr
            self.y_tr = y_tr
        self.train_mae_infer = []

    def on_epoch_end(self, _epoch, _logs=None):
        y_pred = self.model.predict(self.x_tr, verbose=0).ravel()
        self.train_mae_infer.append(float(np.mean(np.abs(y_pred - self.y_tr))))


# ============================================================
# RECRIA O MODELO COM OS MELHORES HPs
# ============================================================
set_reproducibility(SEED)
fixed_trial = optuna.trial.FixedTrial(study.best_params)
best_model, best_bs = build_cnn_from_trial(
    fixed_trial, n_wave=n_wave, seed=SEED,
)
best_model.summary()


In [ ]:
# ============================================================
# TREINO FINAL — agora com FINAL_EPOCHS (maior que EPOCHS_PER_TRIAL)
# ============================================================
# A diferenca pro treino dentro do Optuna:
#   - Mais epocas (sabemos que esses HPs sao bons, vale treinar ate convergir)
#   - patience maior no EarlyStopping (15 vs 8): da mais chance de superar plateaus
#   - ReduceLROnPlateau mais agressivo (cooldown maior): refina o lr no fim

# Sample weights por z (se use_z_weights=True). Default False -> sw_train_final = None.
sw_train_final = make_redshift_weights(
    y_train,
    **{k: WEIGHTS_CONFIG[k] for k in
        ["z_focus_low", "z_focus_high", "w_low", "w_focus", "w_high", "w_max"]},
    enabled=WEIGHTS_CONFIG["use_z_weights"],
)


# Callback custom que mede MAE no train EM MODO INFERENCIA (dropout OFF) a cada
# epoca. history.history["loss"] sempre tem dropout ON (e a loss usada pelos
# gradientes). O GAP entre "loss (dropout on)" e "MAE (dropout off)" e diagnostico:
#   - Sem gap: dropout esta sendo bem absorvido, modelo robusto
#   - Gap grande: dropout esta "escondendo" capacidade — sinal de overfit latente
train_inf_cb = TrainInferenceMAECallback(
    X_train, y_train,
    max_eval_samples=len(y_test),   # nao avaliar nos 120k de train (lento) — sample do tamanho do test
    seed=SEED,
)


callbacks_final = [
    train_inf_cb,
    # EarlyStopping: para se val_loss nao melhorar em 15 epocas seguidas.
    #   restore_best_weights=True: ao parar, volta pesos da melhor epoca
    #   (sem isso, ficaria com os pesos da ultima epoca, que e overfit).
    keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=15, restore_best_weights=True, min_delta=TRAINING_CONFIG["early_stop_min_delta"],
    ),
    # ReduceLROnPlateau: se val_loss estagnar 7 epocas, reduz lr * 0.5.
    #   cooldown=2: depois de reduzir, espera 2 epocas antes de reduzir de novo
    #   (evita reduzir lr todo turno sem dar tempo do modelo se ajustar).
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=TRAINING_CONFIG["reduce_lr_factor"],
        patience=7, min_lr=TRAINING_CONFIG["reduce_lr_min"], cooldown=2,
    ),
]


# Treino. verbose=2: 1 linha por epoca (melhor que verbose=1 = barra de progresso
# que polui o log do papermill).
t0 = time.time()
history_final = best_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=FINAL_EPOCHS,
    batch_size=best_bs,
    sample_weight=sw_train_final,
    callbacks=callbacks_final,
    verbose=2,
)
train_time = time.time() - t0
print(f"\nTempo de treino final: {train_time/60:.1f} min")


## 14. Avaliacao no TEST


In [ ]:
y_pred_test = best_model.predict(X_test, verbose=0).ravel()

dz_test = delta_z_norm(y_pred_test, y_test)

test_metrics = {
    "mae":               float(np.mean(np.abs(y_pred_test - y_test))),
    "bias":              bias(dz_test),
    "sigma_nmad":        sigma_NMAD(dz_test),
    "eta_outliers_0.05": eta_outliers(dz_test, 0.05),
    "eta_outliers_0.15": eta_outliers(dz_test, 0.15),
    "f_inliers_0.15":    f_inliers(dz_test, 0.15),
}

print("=" * 60)
print(f"RESULTADO FINAL — TEST  (n={len(y_test):,})")
print("=" * 60)
for k, v in test_metrics.items():
    print(f"  {k:>22} = {v:.4f}")


## 15. Plots do modelo final


In [ ]:
mae_test        = test_metrics["mae"]
bias_test       = test_metrics["bias"]
sigma_nmad_test = test_metrics["sigma_nmad"]
out_05_test     = test_metrics["eta_outliers_0.05"]
out_15_test     = test_metrics["eta_outliers_0.15"]


fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# (a) curvas
loss_fit = np.asarray(history_final.history["loss"], dtype=float)
val_fit  = np.asarray(history_final.history["val_loss"], dtype=float)
mae_inf  = np.asarray(train_inf_cb.train_mae_infer, dtype=float)

axes[0, 0].plot(loss_fit, label="train loss (dropout on)", lw=2)
axes[0, 0].plot(val_fit,  label="val loss",                lw=2)
if len(mae_inf):
    axes[0, 0].plot(mae_inf, label="train MAE (dropout off)", lw=2, ls="--")
axes[0, 0].set_xlabel("Epoch"); axes[0, 0].set_ylabel("Loss / MAE")
axes[0, 0].set_title("(a) Curvas de treino — gap dropout on/off sinaliza overfit")
axes[0, 0].set_yscale("log"); axes[0, 0].legend()

# (b) z_pred vs z_real
hb = axes[0, 1].hexbin(y_test, y_pred_test, gridsize=40, cmap="viridis", mincnt=1)
axes[0, 1].plot([y_test.min(), y_test.max()],
                [y_test.min(), y_test.max()], "r--", lw=1.5, label="ideal (y=x)")
axes[0, 1].set_xlabel("z_real"); axes[0, 1].set_ylabel("z_predito")
axes[0, 1].set_title(f"(b) z_pred vs z_real  (sigma_NMAD = {sigma_nmad_test:.4f})")
axes[0, 1].legend(); plt.colorbar(hb, ax=axes[0, 1], label="# galaxias")

# (c) hist delta_z
axes[1, 0].hist(np.clip(dz_test, -0.3, 0.3), bins=80,
                color="steelblue", alpha=0.85, edgecolor="black", lw=0.3)
axes[1, 0].axvline(bias_test, color="red", lw=2, label=f"bias = {bias_test:+.4f}")
axes[1, 0].axvline(bias_test + sigma_nmad_test, color="orange", lw=1.2, ls="--",
                   label=f"+/-sigma_NMAD = {sigma_nmad_test:.4f}")
axes[1, 0].axvline(bias_test - sigma_nmad_test, color="orange", lw=1.2, ls="--")
axes[1, 0].axvline(+0.15, color="darkred", lw=1.2, ls=":", label="+/-0.15")
axes[1, 0].axvline(-0.15, color="darkred", lw=1.2, ls=":")
axes[1, 0].set_xlabel("delta_z / (1 + z)"); axes[1, 0].set_ylabel("# galaxias")
axes[1, 0].set_title(f"(c) Residuos  (outliers |>0.15|: {out_15_test:.2f}%)")
axes[1, 0].legend(fontsize=9, loc="upper right"); axes[1, 0].set_xlim(-0.3, 0.3)

# (d) residuos vs z_real
hb2 = axes[1, 1].hexbin(y_test, np.clip(dz_test, -0.3, 0.3),
                        gridsize=40, cmap="magma", mincnt=1)
axes[1, 1].axhline(0,                color="white", lw=1.5, alpha=0.7)
axes[1, 1].axhline(+sigma_nmad_test, color="orange", lw=1.2, ls="--", label="+/-sigma_NMAD")
axes[1, 1].axhline(-sigma_nmad_test, color="orange", lw=1.2, ls="--")
axes[1, 1].axhline(+0.15, color="red", lw=1.2, ls=":", label="+/-0.15")
axes[1, 1].axhline(-0.15, color="red", lw=1.2, ls=":")
axes[1, 1].set_xlabel("z_real"); axes[1, 1].set_ylabel("delta_z / (1 + z)")
axes[1, 1].set_title("(d) Residuos vs z_real"); axes[1, 1].set_ylim(-0.3, 0.3)
axes[1, 1].legend(fontsize=9, loc="upper right")
plt.colorbar(hb2, ax=axes[1, 1], label="# galaxias")

plt.tight_layout()
plt.savefig(PLOT_DIR / "model_final_summary.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Salvo: {PLOT_DIR / 'model_final_summary.png'}")


## 16. Salvar tudo

### Pra inferencia em galaxias novas (modelo NAO usa scalars):
1. Carregar o modelo `.keras`
2. Normalizar o espectro novo com `normalize_spectra`
3. Chamar `model.predict(x_normalizado)` — basta o espectro

### Pra analise dos scalars (info auxiliar nao usada no treino):
- `test_outputs.npz` tem `S_test` (log10, fisicamente interpretavel)
- `run_info.json` lista os scalars e suas features


In [ ]:
import json as _j


# ============================================================
# MODELO — .keras serializa o grafo + pesos
# ============================================================
# .keras eh o formato moderno (substitui o .h5). Inclui ScaledSoftplus
# automaticamente porque registramos com @register_keras_serializable.
best_model.save(MODEL_RUN_DIR / "best_final.keras")


# ============================================================
# SPLITS — indices dos arrays originais (X_spec, y_all)
# ============================================================
# np.savez salva varios arrays num arquivo .npz (zip).
# Argumentos nomeados viram chaves: ao carregar, acessa via npz["pool_idx"].
# Util pra reproduzir o EXATO mesmo split em outro lugar.
np.savez(
    RUN_DIR / "split_idx.npz",
    pool_idx=pool_idx, test_idx=test_idx,
    train_in_pool=train_in_pool, val_in_pool=val_in_pool,
)


# ============================================================
# PREDICOES + SCALARS NO TEST — pra analise posterior
# ============================================================
# Salva tudo do test pra plotar/analisar sem precisar re-rodar o modelo.
# np.array(SCALARS_CONFIG_FEATURES): converte lista Python pra array numpy
# (necessario porque np.savez so aceita arrays).
np.savez(
    RUN_DIR / "test_outputs.npz",
    y_test=y_test, y_pred=y_pred_test, delta_z=dz_test,
    S_test=S_test,
    feature_names=np.array(SCALARS_CONFIG_FEATURES),
)


# ============================================================
# RUN_INFO — metadata completa
# ============================================================
run_info = {
    "created_utc":   datetime.now(timezone.utc).isoformat().replace("+00:00", "Z"),
    "run_tag":       RUN_TAG,
    "config_hash":   config_hash,
    "study_name":    STUDY_NAME,
    "seed":          int(SEED),
    "object_type":   OBJECT_TYPE,
    "n_wave":        int(n_wave),

    "splits": {
        "test_size":  float(TRAINING_CONFIG["test_size"]),
        "val_size":   float(TRAINING_CONFIG["val_size"]),
        "n_train":    int(len(y_train)),
        "n_val":      int(len(y_val)),
        "n_test":     int(len(y_test)),
        "q_outer":    int(q_outer),
        "q_inner":    int(q_inner),
    },

    "scalars": {
        "features":  SCALARS_CONFIG_FEATURES,
        "n_scalars": int(n_scalars),
        "note":      "scalars salvos em log10 (sem z-score), nao alimentam o modelo",
    },

    "objective_mode":   OBJECTIVE_CONFIG["mode"],
    "w_sigma":          float(OBJECTIVE_CONFIG["w_sigma"]),
    "sigma_nmad_ref":   float(sigma_NMAD_ref),
    "eta_outliers_ref": float(eta_outliers_ref),

    "n_trials":          int(len(study.trials)),
    "n_trials_complete": int(n_complete),
    "n_trials_pruned":   int(n_pruned),
    "best_trial":        int(study.best_trial.number),
    "best_score":        float(study.best_value),
    "best_params":       study.best_trial.params,
    "best_user_attrs":   dict(study.best_trial.user_attrs),

    "final_epochs_trained": int(len(history_final.history["loss"])),
    "test_metrics":         test_metrics,
    "train_time_s":         float(train_time),

    "training_config":  TRAINING_CONFIG,
    "objective_config": OBJECTIVE_CONFIG,
    "hpo_config":       HPO_CONFIG,
    "hp_space":         HP_SPACE,
    "scalars_config":   SCALARS_CONFIG,
    "weights_config":   WEIGHTS_CONFIG,

    "env": {
        "python":  sys.version.split()[0],
        "tf":      tf.__version__,
        "keras":   keras.__version__,
        "numpy":   np.__version__,
        "sklearn": sklearn.__version__,
        "optuna":  optuna.__version__,
    },
}

with open(RUN_DIR / "run_info.json", "w") as f:
    _j.dump(run_info, f, indent=2, default=str)


print(f"Salvo em:")
print(f"  modelo         : {MODEL_RUN_DIR / 'best_final.keras'}")
print(f"  splits         : {RUN_DIR / 'split_idx.npz'}")
print(f"  preds          : {RUN_DIR / 'test_outputs.npz'}")
print(f"  run_info       : {RUN_DIR / 'run_info.json'}")
print(f"  top trials     : {RUN_DIR / 'optuna_top.csv'}")
print(f"  plots em       : {PLOT_DIR}/")
print(f"  optuna DB      : {DB_PATH}")


## 17. Sanity check — reload + predicao identica


In [ ]:
# ============================================================
# SANITY CHECK: o modelo salvo carrega e prediz IGUAL ao original?
# ============================================================
# Por que isso eh importante?
#   - .keras serialization pode falhar com custom layers (ex: ScaledSoftplus)
#   - Se algo der ruim no save/load, o modelo "reloaded" pode prever errado
#   - Esse assert garante que o .keras salvo eh fiel ao modelo treinado
#
# Tolerancia 1e-5: diferencas de float32 podem dar oscilacoes minusculas, mas
# nada acima disso e aceitavel.

reloaded = keras.models.load_model(MODEL_RUN_DIR / "best_final.keras")
y_check = reloaded.predict(X_test[:100], verbose=0).ravel()
max_delta = float(np.abs(y_check - y_pred_test[:100]).max())

print(f"max |delta| (reloaded vs original): {max_delta:.2e}")
# atol=1e-4: round-trip float32 + nao-determinismo de kernels conv na GPU dao
# |delta| ~1e-5 mesmo com o modelo 100% correto (visto: 2.05e-05 no job 39880,
# que so falhou por isso). 1e-4 pega ScaledSoftplus NAO registrada (delta ~0.1-1)
# sem barrar o ruido numerico benigno.
assert np.allclose(y_check, y_pred_test[:100], atol=1e-4), \
    "Reload divergiu MUITO (>1e-4) — custom layer (ScaledSoftplus) provavelmente nao registrada. Investigar."
print("Reload OK.")
